In [ ]:
import requests
import json
from datetime import date

# Fetch
url = "https://api.open-meteo.com/v1/forecast?latitude=39.04&longitude=-77.49&daily=temperature_2m_max,temperature_2m_min,precipitation_sum,windspeed_10m_max,weathercode&timezone=America%2FNew_York&forecast_days=7&temperature_unit=fahrenheit&windspeed_unit=mph&precipitation_unit=inch"

response = requests.get(url)
data = response.json()

# Flatten
daily = data["daily"]
rows = []
for i in range(len(daily["time"])):
    rows.append({
        "date": daily["time"][i],
        "temp_max_f": daily["temperature_2m_max"][i],
        "temp_min_f": daily["temperature_2m_min"][i],
        "precipitation_in": daily["precipitation_sum"][i],
        "windspeed_max_mph": daily["windspeed_10m_max"][i],
        "weather_code": daily["weathercode"][i],
        "location": "Ashburn-VA",
        "ingested_at": str(date.today())
    })

# Write to lh_bronze
df = spark.createDataFrame(rows)
spark.sql("DROP TABLE IF EXISTS weather_ashburn")
df.write.format("delta").mode("overwrite").saveAsTable("weather_ashburn")

print(f"✅ {df.count()} rows written to lh_bronze.weather_ashburn")

StatementMeta(, , -1, SessionError, , SessionError, True)

InvalidHttpRequest: [TooManyRequestsForCapacity] [TooManyRequestsForCapacity] HTTP Response code 430: This Spark job can’t be run because you’ve hit a Spark compute or API rate limit. To proceed, cancel an active Spark job through the Monitoring hub, choose a larger capacity SKU, or try again later. For more visibility and control, go to Workspace settings → Job management (Job Concurrency & Queue Monitoring) to review running and queued Spark jobs, understand capacity contention, and take action as needed. [Learn more at 'https://go.microsoft.com/fwlink/?linkid=2356970&clcid=0x409']. HTTP status code: 430.

In [ ]:
import requests
import json
from datetime import date

# Historical archive API - same as forecast but different endpoint and date range
url = "https://archive-api.open-meteo.com/v1/archive?latitude=39.04&longitude=-77.49&start_date=2025-01-20&end_date=2025-01-24&daily=temperature_2m_max,temperature_2m_min,precipitation_sum,windspeed_10m_max,weathercode&timezone=America%2FNew_York&temperature_unit=fahrenheit&windspeed_unit=mph&precipitation_unit=inch"

response = requests.get(url)
data = response.json()

# Flatten
daily = data["daily"]
rows = []
for i in range(len(daily["time"])):
    rows.append({
        "date": daily["time"][i],
        "temp_max_f": daily["temperature_2m_max"][i],
        "temp_min_f": daily["temperature_2m_min"][i],
        "precipitation_in": daily["precipitation_sum"][i],
        "windspeed_max_mph": daily["windspeed_10m_max"][i],
        "weather_code": daily["weathercode"][i],
        "location": "Ashburn-VA",
        "ingested_at": str(date.today())
    })

# Append to existing Bronze weather table
df = spark.createDataFrame(rows)
df.write.format("delta").mode("append").saveAsTable("weather_ashburn")

print(f"✅ {df.count()} historical rows appended to lh_bronze.weather_ashburn")

StatementMeta(, , -1, Cancelled, , Cancelled, True)

In [ ]:
import requests
from datetime import date

# Summer historical weather - triggers heat stress categories
url = "https://archive-api.open-meteo.com/v1/archive?latitude=39.04&longitude=-77.49&start_date=2025-07-15&end_date=2025-07-16&daily=temperature_2m_max,temperature_2m_min,precipitation_sum,windspeed_10m_max,weathercode&timezone=America%2FNew_York&temperature_unit=fahrenheit&windspeed_unit=mph&precipitation_unit=inch"

response = requests.get(url)
data = response.json()

daily = data["daily"]
rows = []
for i in range(len(daily["time"])):
    rows.append({
        "date": daily["time"][i],
        "temp_max_f": daily["temperature_2m_max"][i],
        "temp_min_f": daily["temperature_2m_min"][i],
        "precipitation_in": daily["precipitation_sum"][i],
        "windspeed_max_mph": daily["windspeed_10m_max"][i],
        "weather_code": daily["weathercode"][i],
        "location": "Ashburn-VA",
        "ingested_at": str(date.today())
    })

df = spark.createDataFrame(rows)
df.write.format("delta").mode("append").saveAsTable("weather_ashburn")
print(f"✅ {df.count()} summer rows appended to lh_bronze.weather_ashburn")

StatementMeta(, , -1, Cancelled, , Cancelled, True)

In [ ]:
import requests
from datetime import date

# Monthly weather snapshots Feb-Jun 2025 to fill time series gap
url = "https://archive-api.open-meteo.com/v1/archive?latitude=39.04&longitude=-77.49&start_date=2025-02-15&end_date=2025-06-15&daily=temperature_2m_max,temperature_2m_min,precipitation_sum,windspeed_10m_max,weathercode&timezone=America%2FNew_York&temperature_unit=fahrenheit&windspeed_unit=mph&precipitation_unit=inch"

response = requests.get(url)
data = response.json()

daily = data["daily"]
rows = []
for i in range(len(daily["time"])):
    # Only keep 15th of each month to match capacity data points
    if daily["time"][i].endswith("-15"):
        rows.append({
            "date": daily["time"][i],
            "temp_max_f": daily["temperature_2m_max"][i],
            "temp_min_f": daily["temperature_2m_min"][i],
            "precipitation_in": daily["precipitation_sum"][i],
            "windspeed_max_mph": daily["windspeed_10m_max"][i],
            "weather_code": daily["weathercode"][i],
            "location": "Ashburn-VA",
            "ingested_at": str(date.today())
        })

df = spark.createDataFrame(rows)
df.write.format("delta").mode("append").saveAsTable("weather_ashburn")
print(f"✅ {df.count()} monthly weather rows appended to lh_bronze.weather_ashburn")

StatementMeta(, , -1, Cancelled, , Cancelled, True)